# Explore simulation run

Load tick log and final world snapshot; plot population, organic heatmap, and concept counts.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "sim-core").exists():
    ROOT = ROOT.parent

EXPORTS = ROOT / "sim-core" / "exports"
TICK_LOG = EXPORTS / "logs" / "tick_log.jsonl"
SNAPSHOT = EXPORTS / "snapshots" / "world_final.json"

print("Tick log:", TICK_LOG)
print("Snapshot:", SNAPSHOT)

In [ ]:
def load_tick_log(path: Path) -> list[dict]:
    entries = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

def load_snapshot(path: Path) -> dict:
    with path.open() as f:
        return json.load(f)

entries = load_tick_log(TICK_LOG) if TICK_LOG.exists() else []
snapshot = load_snapshot(SNAPSHOT) if SNAPSHOT.exists() else {}
print(f"Loaded {len(entries)} tick log entries, snapshot time={snapshot.get('time')}")

In [ ]:
if entries:
    pop = [len(e.get("creatures", [])) for e in entries]
    ticks = [e.get("tick", i) for i, e in enumerate(entries)]
    births = [len(e.get("births", [])) for e in entries]
    deaths = [len(e.get("deaths", [])) for e in entries]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ticks, pop, label="population")
    ax.plot(ticks, births, alpha=0.5, label="births/tick")
    ax.plot(ticks, deaths, alpha=0.5, label="deaths/tick")
    ax.set_xlabel("tick")
    ax.set_ylabel("count")
    ax.set_title("Population over time")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No tick log — run: cargo run -- --ticks 1000 --seed 42 --world-size 4 --creatures 12 --output exports")

In [ ]:
chunks = snapshot.get("chunks", [])
if chunks:
    c0 = chunks[0]
    organic = c0.get("organic", [])
    if organic:
        import numpy as np
        grid = np.array(organic)
        fig, ax = plt.subplots(figsize=(5, 5))
        im = ax.imshow(grid.T, origin="lower", cmap="YlGn")
        ax.set_title(f"Organic slice z={c0.get('slice_z')} chunk {c0.get('coord')}")
        plt.colorbar(im, ax=ax, label="organic")
        plt.tight_layout()
        plt.show()
else:
    print("No chunk data in snapshot")

In [ ]:
creatures = snapshot.get("creatures", [])
if creatures:
    df = pd.DataFrame([
        {
            "id": c["id"],
            "concept_count": c.get("concept_count", 0),
            "trusted_signatures": c.get("trusted_signature_count", 0),
            "memory_nodes": c.get("memory_node_count", 0),
        }
        for c in creatures
    ])
    display(df.describe())

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(df["id"].astype(str), df["concept_count"], label="concepts")
    if df["trusted_signatures"].sum() > 0:
        ax.bar(df["id"].astype(str), df["trusted_signatures"], alpha=0.6, label="trusted signatures")
    ax.set_xlabel("creature id")
    ax.set_ylabel("count")
    ax.set_title("Concept and trusted-signature counts (final snapshot)")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No creatures in snapshot")